# Task 01: Data preparation and baseline

**Goal:** prepare a public dataset and get a first, deliberately simple model working.

**Dataset:** Breast Cancer Wisconsin (Diagnostic), from the UCI Machine Learning Repository. It ships with scikit-learn, so no download is needed.
**Task:** binary classification. Predict whether a tumour is malignant or benign from 30 numeric measurements.

**Plan**
1. Load the data and take a quick look
2. Split into train and test **before anything else**
3. Build a trivial baseline (always predict the most common class)
4. Train one simple model (scaled logistic regression)
5. Report one accuracy number on the held-out test set

In [8]:
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix

RANDOM_STATE = 42  # fixed so the result is reproducible

## 1. Load the data

In [9]:
data = load_breast_cancer(as_frame=True)
df = data.frame
print("Shape:", df.shape)
df.head()

Shape: (569, 31)


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0


In this dataset `target = 0` means **malignant** and `target = 1` means **benign**.
A quick check for missing values, duplicates and class balance:

In [10]:
print("Missing values:", df.isna().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
print()
print("Class balance:")
print(df["target"].map({0: "malignant", 1: "benign"}).value_counts(normalize=True).round(3))

Missing values: 0
Duplicate rows: 0

Class balance:
target
benign       0.627
malignant    0.373
Name: proportion, dtype: float64


No missing values and no duplicates, so no cleaning is needed. The classes are imbalanced (about 63% benign), which matters for the baseline below.

## 2. Train/test split (done first)

The split happens before any scaling, fitting or tuning, so the test set never influences a decision. I use `stratify=y` so both sets keep the same class balance.

In [11]:
X = df.drop(columns="target")
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Share of benign in train:", round(y_train.mean(), 3))
print("Share of benign in test: ", round(y_test.mean(), 3))

Train: (455, 30) | Test: (114, 30)
Share of benign in train: 0.626
Share of benign in test:  0.632


## 3. Trivial baseline

The baseline always predicts the most frequent class from the training data. Any real model has to beat this, otherwise it has learned nothing.

In [12]:
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)

baseline_acc = accuracy_score(y_test, baseline.predict(X_test))
print(f"Baseline accuracy: {baseline_acc:.3f}")

Baseline accuracy: 0.632


## 4. First simple model

Logistic regression inside a pipeline with `StandardScaler`. Using a pipeline means the scaler is fitted on the training data only, so no information leaks from the test set.

In [13]:
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
model_acc = accuracy_score(y_test, y_pred)
print(f"Logistic regression accuracy: {model_acc:.3f}")

Logistic regression accuracy: 0.982


## 5. Result

In [14]:
print(f"Baseline (most frequent class): {baseline_acc:.3f}")
print(f"Logistic regression:            {model_acc:.3f}")
print(f"Improvement over baseline:      {model_acc - baseline_acc:+.3f}")
print()
print("Confusion matrix (rows = actual, columns = predicted; 0 = malignant, 1 = benign):")
print(pd.DataFrame(confusion_matrix(y_test, y_pred),
                   index=["actual 0", "actual 1"],
                   columns=["pred 0", "pred 1"]))

Baseline (most frequent class): 0.632
Logistic regression:            0.982
Improvement over baseline:      +0.351

Confusion matrix (rows = actual, columns = predicted; 0 = malignant, 1 = benign):
          pred 0  pred 1
actual 0      41       1
actual 1       1      71


## Notes and decisions

- **Split first.** I split 80/20 before looking at anything else and used the test set exactly once, for the final numbers.
- **Stratified split.** The classes are imbalanced, so stratifying keeps the test set representative.
- **Why this baseline.** Because about 63% of the rows are benign, a model that always says "benign" scores about 63% without learning anything. That is the number to beat, not 50%.
- **The one number I trust:** test accuracy of the scaled logistic regression, reported above. It comes from data the model never saw during training, with a fixed random seed.
- **Limitation.** The test set is only 114 rows, so one extra mistake moves accuracy by about 0.9 points. Treat the result as approximate. Cross-validation would give a steadier estimate and is a natural next step.
- **Kept out on purpose:** no hyperparameter tuning, no model comparison, no feature engineering. This task is only about a clean baseline.